# 04 · Modelli sulle forme d'onda (CNN 1D)

**Perché.** Nelle prime prove (notebook 00) le 7 misure tabulari portano poco segnale. In EchoNext il segnale principale sta nella forma d'onda a 12 derivazioni (10 s a 250 Hz, array `N × 1 × 2500 × 12`). Questo notebook addestra una rete convoluzionale 1D, standard per ECG, e la confronta con la baseline tabulare.

**Scelte di progetto**
- **Multi-etichetta**: una sola rete predice tutte le 12 etichette (composito + singole condizioni) con una loss che ignora le etichette mancanti. Le condizioni condividono la rappresentazione e si vede subito per quali la forma d'onda funziona.
- **Fusione con le feature tabulari** (età, sesso, ecc.) dopo il pooling: opzionale (`USE_TABULAR`).
- Valutazione su validation con IC95% e per fascia di QRS; test solo alla fine.
- Si salvano predizioni ed embedding di validation in `OUT_DIR`, riutilizzabili per un clustering su una rappresentazione più ricca delle misure tabulari.

**Prima di iniziare:** serve `torch`. Con una GPU NVIDIA (versione CUDA di torch) un'epoca richiede pochi minuti; su sola CPU è molto più lento, quindi per una prova usa `DEBUG_N` nella cella di setup (es. 2000). I file `.npy` delle forme d'onda (circa 12 GB) sono letti dal disco con `mmap`, quindi un SSD aiuta.

Riferimento utile: il repository ufficiale **IntroECG** (github.com/PierreElias/IntroECG) contiene il modello Mini-Model pubblicato con pesi pre-addestrati, utile come confronto.

In [ ]:
#@title Setup comune: configurazione, caricamento dati, funzioni di utilità
import os, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")

# ===================== CONFIGURAZIONE =====================
DATA_DIR = Path(os.environ.get("ECHONEXT_DIR", "/home/teo/Desktop/dataset"))   # cartella con lo zip (o i file già estratti)
OUT_DIR  = Path(os.environ.get("ECHONEXT_OUT", str(Path.home() / "echonext_results")))  # dove salvare i risultati
SEED = 42
DEBUG_N = int(os.environ.get("ECHONEXT_DEBUG_N", "0")) or None  # es. 3000 = prova rapida con 3000 righe per split
RUN_FINAL_TEST = False   # lasciare False finché pipeline e scelte non sono congelate (vedi ultima cella)
TARGET = "shd_moderate_or_greater_flag"
# ==========================================================

def locate_dataset(root):
    """Trova il CSV dei metadati (anche in sottocartelle, anche .csv.gz); se c'è solo uno zip lo estrae in <root>/extracted.
    Se non trova nulla, stampa il contenuto della cartella per capire cosa manca."""
    import zipfile
    root = Path(root).expanduser()
    if not root.exists(): raise FileNotFoundError(f"Cartella non trovata: {root}")
    def find_csv(base):
        c = [p for p in base.rglob("*") if p.is_file() and "metadata" in p.name.lower() and p.name.lower().endswith((".csv", ".csv.gz"))]
        return sorted(c, key=lambda p: ("100k" not in p.name.lower(), len(str(p))))
    hits = find_csv(root)
    if hits: return hits[0]
    target = root / "extracted"
    for z in sorted(root.rglob("*.zip")):
        with zipfile.ZipFile(z) as zf:
            if any("metadata" in n.lower() and ".csv" in n.lower() for n in zf.namelist()):
                print(f"estraggo {z.name} in {target} ... (può richiedere minuti)"); zf.extractall(target); break
    hits = find_csv(target) if target.exists() else []
    if hits: return hits[0]
    files = sorted(p for p in root.rglob("*") if p.is_file())
    listing = "\n".join(f"  {p.relative_to(root)}  ({p.stat().st_size / 1e6:.2f} MB)" for p in files[:30])
    raise FileNotFoundError(f"Nessun CSV di metadata in {root}.\nContenuto trovato ({len(files)} file):\n{listing or '  (nessun file)'}\n"
                            "Se vedi solo index.html o file di pochi KB il download non è andato a buon fine (credenziali o DUA non firmato); "
                            "se i file stanno altrove, punta DATA_DIR alla cartella giusta.")

META_PATH = locate_dataset(DATA_DIR)
DATA_DIR = META_PATH.parent
print("dati in:", DATA_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)

# Feature ECG ammesse come INPUT dei modelli (misure automatiche dell'ECG + demografia)
ECG_FEATURES = ["sex", "ventricular_rate", "atrial_rate", "pr_interval", "qrs_duration", "qt_corrected", "age_at_ecg"]
# Etichette binarie derivate dall'ecocardiogramma (possibili TARGET)
FLAG_COLS = ["shd_moderate_or_greater_flag", "lvef_lte_45_flag", "lvwt_gte_13_flag",
             "aortic_stenosis_moderate_or_greater_flag", "aortic_regurgitation_moderate_or_greater_flag",
             "mitral_regurgitation_moderate_or_greater_flag", "tricuspid_regurgitation_moderate_or_greater_flag",
             "pulmonary_regurgitation_moderate_or_greater_flag", "rv_systolic_dysfunction_moderate_or_greater_flag",
             "pericardial_effusion_moderate_large_flag", "pasp_gte_45_flag", "tr_max_gte_32_flag"]
# Colonne derivate dall'eco: MAI usarle come input (leakage del target)
ECHO_ONLY_COLS = ["aortic_stenosis_value", "aortic_regurgitation_value", "mitral_regurgitation_value",
                  "tricuspid_regurgitation_value", "pulmonary_regurgitation_value", "rv_systolic_function_value",
                  "pericardial_effusion_value", "ivs_measurement", "lvpw_measurement", "pasp_value",
                  "tr_max_velocity_value", "lvef_value"] + FLAG_COLS
assert not set(ECG_FEATURES) & set(ECHO_ONLY_COLS), "leakage: feature eco tra gli input"

# ---------- caricamento metadata ----------
meta = pd.read_csv(META_PATH)
meta["sex"] = meta["sex"].astype(str).str.lower().map({"male": 1, "female": 0, "m": 1, "f": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0})
assert meta["sex"].notna().mean() > 0.9, "colonna sex non interpretata: controlla i valori"
# l'ordine delle righe del CSV corrisponde all'ordine dei file .npy di ciascuno split
meta["row_in_split"] = meta.groupby("split").cumcount()
if DEBUG_N:
    meta = meta.groupby("split").head(DEBUG_N)
train, val, test = (meta[meta.split == s].copy() for s in ("train", "val", "test"))
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}  (no_split={int((meta.split=='no_split').sum()):,})")

# ---------- metriche ----------
from sklearn.metrics import roc_auc_score, average_precision_score, mean_absolute_error, r2_score

def safe_auc(y, p):
    y = np.asarray(y)
    return roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan

def boot_ci(y, p, fn=safe_auc, n=300, seed=0):
    """Intervallo di confidenza al 95% per bootstrap sulle righe (val/test hanno 1 ECG per paziente)."""
    rng = np.random.default_rng(seed); y = np.asarray(y); p = np.asarray(p); vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y)); v = fn(y[i], p[i])
        if not np.isnan(v): vals.append(v)
    return (np.percentile(vals, 2.5), np.percentile(vals, 97.5)) if vals else (np.nan, np.nan)

def fmt_auc(y, p):
    lo, hi = boot_ci(y, p)
    return safe_auc(y, p), lo, hi

# ---------- dati e modelli ----------
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict

def get_xy(df, target, features=ECG_FEATURES):
    """Restituisce X, y, df filtrato (righe con target mancante escluse)."""
    d = df.dropna(subset=[target])
    y = d[target].values
    y = y.astype(int) if target in FLAG_COLS else y.astype(float)
    return d[list(features)].astype(float), y, d

def model_zoo(seed=SEED):
    """Più modelli da confrontare: lineare, lineare sparso (Lasso), foresta, boosting, rete densa (MLP)."""
    imp = lambda: SimpleImputer(strategy="median", add_indicator=True)  # l'indicatore cattura "PR/atrial rate mancante" (es. fibrillazione)
    return {
        "logreg": make_pipeline(imp(), StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)),
        "lasso_logreg": make_pipeline(imp(), StandardScaler(), LogisticRegression(penalty="l1", solver="liblinear", C=0.1)),
        "random_forest": make_pipeline(imp(), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=seed)),
        "hist_gb": HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=250, random_state=seed),
        "mlp_dense": make_pipeline(imp(), StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, early_stopping=True, max_iter=200, random_state=seed)),
    }

def eval_models(Xtr, ytr, Xva, yva, models=None):
    """Addestra su train, valuta su val. Ritorna tabella AUROC (con IC95%) e predizioni."""
    rows, preds = [], {}
    for name, m in (models or model_zoo()).items():
        t0 = time.time(); m.fit(Xtr, ytr); p = m.predict_proba(Xva)[:, 1]
        a, lo, hi = fmt_auc(yva, p)
        rows.append(dict(model=name, AUROC=a, CI_lo=lo, CI_hi=hi, AUPRC=average_precision_score(yva, p),
                         prev_val=float(np.mean(yva)), n_val=len(yva), sec=round(time.time() - t0, 1)))
        preds[name] = p
    return pd.DataFrame(rows).sort_values("AUROC", ascending=False).reset_index(drop=True), preds

def oof_proba(model, X, y, groups, n_splits=5):
    """Predizioni out-of-fold con CV raggruppata per paziente (nessun paziente in train e test dello stesso fold)."""
    return cross_val_predict(model, X, y, groups=groups, cv=GroupKFold(n_splits), method="predict_proba")[:, 1]

def add_groups(df):
    """Aggiunge fasce di QRS ed età, usate per stratificare."""
    d = df.copy()
    d["qrs_bin"] = pd.cut(d.qrs_duration, [0, 100, 120, 10000], labels=["<100", "100-119", ">=120"], right=False).astype(object)
    d["age_bin"] = pd.cut(d.age_at_ecg, [0, 50, 65, 75, 200], labels=["<50", "50-64", "65-74", ">=75"], right=False).astype(object)
    return d

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(senza GPU l'addestramento sarà molto lento)")
torch.manual_seed(SEED)

# ---- parametri ----
EPOCHS, BATCH, LR = 12, 128, 2e-3
USE_TABULAR = True
TARGETS = [TARGET] + [c for c in FLAG_COLS if c != TARGET]
WAVE_DIR = DATA_DIR
print("forme d'onda lette da:", WAVE_DIR)

## 1 · Dataset

Le forme d'onda sono già preprocessate dal dataset (filtro mediano, clipping, normalizzazione globale). Si leggono con `mmap` per non caricare 12 GB in RAM. Le feature tabulari vengono imputate con la mediana e standardizzate con le statistiche del **train**.

In [ ]:
imp_t = SimpleImputer(strategy="median").fit(train[ECG_FEATURES].astype(float))
sc_t = StandardScaler().fit(imp_t.transform(train[ECG_FEATURES].astype(float)))
def tab(df): return np.clip(sc_t.transform(imp_t.transform(df[ECG_FEATURES].astype(float))), -5, 5).astype(np.float32)

class ECGSet(Dataset):
    def __init__(self, split, df):
        self.path = WAVE_DIR / f"EchoNext_{split}_waveforms.npy"; self.W = None
        self.rows = df.row_in_split.values; self.y = df[TARGETS].values.astype(np.float32); self.t = tab(df)
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        if self.W is None: self.W = np.load(self.path, mmap_mode="r")     # un memmap per worker
        x = np.ascontiguousarray(self.W[self.rows[i], 0].T)                 # (12, 2500)
        return torch.from_numpy(x), torch.from_numpy(self.t[i]), torch.from_numpy(self.y[i])

NW = 2 if device == "cuda" else 0
ds = {s: ECGSet(s, d) for s, d in [("train", train), ("val", val), ("test", test)]}
dl_train = DataLoader(ds["train"], batch_size=BATCH, shuffle=True, num_workers=NW, drop_last=True, pin_memory=device == "cuda")
dl_eval = lambda s: DataLoader(ds[s], batch_size=256, shuffle=False, num_workers=NW)
x0, t0_, y0 = ds["train"][0]; print("esempio:", tuple(x0.shape), tuple(t0_.shape), tuple(y0.shape))

## 2 · Modello

ResNet 1D compatta (circa 0.5 M parametri): convoluzione iniziale, quattro stadi residui con downsampling, pooling medio+massimo, fusione opzionale con le feature tabulari, una testa con un'uscita per etichetta.

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv1d(cin, cout, 7, stride, 3, bias=False); self.b1 = nn.BatchNorm1d(cout)
        self.c2 = nn.Conv1d(cout, cout, 7, 1, 3, bias=False); self.b2 = nn.BatchNorm1d(cout)
        self.sk = nn.Sequential(nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout)) if (stride != 1 or cin != cout) else nn.Identity()
    def forward(self, x):
        h = F.relu(self.b1(self.c1(x))); h = self.b2(self.c2(h)); return F.relu(h + self.sk(x))

class ECGNet(nn.Module):
    def __init__(self, n_out, n_tab, use_tab):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(12, 32, 15, 2, 7, bias=False), nn.BatchNorm1d(32), nn.ReLU())
        self.body = nn.Sequential(Block(32, 32, 2), Block(32, 64, 2), Block(64, 128, 2), Block(128, 128, 2))
        self.use_tab = use_tab; emb = 256 + (32 if use_tab else 0)
        self.tab = nn.Sequential(nn.Linear(n_tab, 32), nn.ReLU()) if use_tab else None
        self.drop = nn.Dropout(0.3); self.head = nn.Linear(emb, n_out)
    def embed(self, x, t):
        h = self.body(self.stem(x)); h = torch.cat([h.mean(-1), h.amax(-1)], 1)
        return torch.cat([h, self.tab(t)], 1) if self.use_tab else h
    def forward(self, x, t, return_emb=False):
        e = self.embed(x, t); o = self.head(self.drop(e)); return (o, e) if return_emb else o

model = ECGNet(len(TARGETS), len(ECG_FEATURES), USE_TABULAR).to(device)
print("parametri:", sum(p.numel() for p in model.parameters()))

def masked_bce(logits, y):
    m = ~torch.isnan(y); y0 = torch.nan_to_num(y)
    l = F.binary_cross_entropy_with_logits(logits, y0, reduction="none")
    return (l * m).sum() / m.sum().clamp(min=1)

## 3 · Addestramento

AdamW con schedule OneCycle, precisione mista se c'è una GPU. A ogni epoca si valuta l'AUROC su validation per il target principale e si conserva il modello migliore. La scelta dell'epoca sul target di validation introduce un piccolo ottimismo: il valore definitivo si legge su test.

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl_train))
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
use_amp = device == "cuda"

@torch.no_grad()
def predict(split, return_emb=False):
    model.eval(); P, E = [], []
    for x, t, y in dl_eval(split):
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            o, e = model(x.to(device), t.to(device), return_emb=True)
        P.append(torch.sigmoid(o.float()).cpu().numpy()); E.append(e.float().cpu().numpy())
    return np.vstack(P), (np.vstack(E) if return_emb else None)

tix = TARGETS.index(TARGET); yv_main = val[TARGET].values; vm = ~np.isnan(yv_main)
best, best_state, hist = -1, None, []
for ep in range(EPOCHS):
    model.train(); tl, t0 = 0.0, time.time()
    for x, t, y in dl_train:
        x, t, y = x.to(device, non_blocking=True), t.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            loss = masked_bce(model(x, t).float(), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); tl += loss.item()
    P, _ = predict("val"); auc = safe_auc(yv_main[vm], P[vm, tix])
    hist.append(dict(epoch=ep + 1, train_loss=tl / len(dl_train), val_AUROC=auc))
    if auc > best: best, best_state = auc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"epoca {ep + 1:2d}/{EPOCHS}  loss={tl / len(dl_train):.4f}  val AUROC({TARGET})={auc:.4f}  ({time.time() - t0:.0f}s)")
model.load_state_dict(best_state); torch.save(best_state, OUT_DIR / "04_cnn_best.pt")
pd.DataFrame(hist).plot(x="epoch", y="val_AUROC", marker="o", figsize=(5, 3)); plt.show()

## 4 · Risultati su validation: per condizione e per fascia di QRS

Confronto tra la CNN sulle forme d'onda e il boosting sulle sole feature tabulari, stesse righe di validation.

In [ ]:
Pva, Eva = predict("val", return_emb=True)
rows = []
for j, tg in enumerate(TARGETS):
    yv = val[tg].values; m = ~np.isnan(yv)
    if yv[m].sum() < 10 or (m.sum() - yv[m].sum()) < 10: continue
    Xtr_, ytr_, _ = get_xy(train, tg); tab_p = model_zoo()["hist_gb"].fit(Xtr_, ytr_).predict_proba(val.loc[m, ECG_FEATURES].astype(float))[:, 1]
    a, lo, hi = fmt_auc(yv[m], Pva[m, j])
    rows.append(dict(target=tg, n_pos_val=int(yv[m].sum()), AUROC_CNN=a, CI_lo=lo, CI_hi=hi, AUROC_tabulare=safe_auc(yv[m], tab_p)))
cnn_res = pd.DataFrame(rows); cnn_res.to_csv(OUT_DIR / "04_cnn_per_condizione.csv", index=False); display(cnn_res.round(3))
cnn_res.set_index("target")[["AUROC_CNN", "AUROC_tabulare"]].sort_values("AUROC_CNN").plot.barh(figsize=(8, 4.5))
plt.axvline(0.5, color="k", ls=":"); plt.xlim(0.4, 1); plt.tight_layout(); plt.show()

# per fascia di QRS, target principale
vg = add_groups(val); rows = []
for g in ["<100", "100-119", ">=120"]:
    m = ((vg.qrs_bin == g).values) & vm
    if m.sum() > 50 and len(np.unique(yv_main[m])) == 2:
        a, lo, hi = fmt_auc(yv_main[m], Pva[m, tix]); rows.append(dict(fascia_QRS=g, n=int(m.sum()), prev=yv_main[m].mean(), AUROC_CNN=a, CI_lo=lo, CI_hi=hi))
display(pd.DataFrame(rows).round(3))

np.savez_compressed(OUT_DIR / "04_val_pred_emb.npz", pred=Pva, emb=Eva, patient_key=val.patient_key.values, row_in_split=val.row_in_split.values, targets=np.array(TARGETS))

**Come leggere.** Se l'AUROC della CNN sul composito e su alcune condizioni supera nettamente quello tabulare, il segnale c'è ed è nella forma d'onda: la strada giusta per i partner è lavorare sulle forme d'onda, usando i sottogruppi (QRS, età) per capire dove funziona meglio. Gli embedding salvati permettono di rifare il clustering su una rappresentazione molto più informativa delle 7 misure tabulari. Gli embedding di train non vengono salvati di proposito: la rete li ha visti con le etichette e sarebbero ottimistici.

## 5 · Valutazione finale su test (una sola volta)

In [ ]:
if RUN_FINAL_TEST:
    Pte, _ = predict("test"); yt = test[TARGET].values; m = ~np.isnan(yt)
    print(f"TEST {TARGET}: AUROC CNN =", [round(v, 3) for v in fmt_auc(yt[m], Pte[m, tix])])
    for j, tg in enumerate(TARGETS):
        y_ = test[tg].values; mm = ~np.isnan(y_)
        if y_[mm].sum() >= 10 and (mm.sum() - y_[mm].sum()) >= 10: print(f"  {tg}: {safe_auc(y_[mm], Pte[mm, j]):.3f}")
else:
    print("RUN_FINAL_TEST=False: test non toccato.")